In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
from pathlib import Path
from matplotlib.gridspec import GridSpec
from scipy.stats import kruskal, mannwhitneyu

from pyriemann.utils.mean import mean_riemann
from pyriemann.utils.distance import distance_riemann

data_path = Path("") # Insert path to the folder containing the .npy files

In [ ]:
# --- Publisher Compliance Settings ---
plt.rcParams.update({
    "pdf.fonttype": 42,      # Ensures fonts are embedded
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "axes.linewidth": 0.75,  # Ensures axes lines > 0.25pt
    "xtick.major.width": 0.75,
    "ytick.major.width": 0.75,
})

# Conversion constant
MM_TO_INCH = 1 / 25.4

def plot_timeline(results, active_clusters, labels_raw, time_idx, color_map):
    """
    Figure 1: Timeline (Half-page width: 85mm)
    """
    # 85mm width, approx 60mm height
    width_in = 85 * MM_TO_INCH
    height_in = 60 * MM_TO_INCH
    
    fig, ax = plt.subplots(figsize=(width_in, height_in), constrained_layout=True)
    
    half_idx = len(time_idx) // 2
    t_half = time_idx[:half_idx]
    l_half = labels_raw[:half_idx]
    
    for k in active_clusters:
        mask = l_half == k
        if np.sum(mask) == 0: continue
        c = color_map.get(str(k), 'gray')
        ax.scatter(t_half[mask], l_half[mask], color=c, 
                label=f"C{k}", alpha=0.8, s=20, edgecolors='white', linewidth=0.5)
        
    ax.set_yticks(active_clusters)
    ax.set_ylabel("Cluster ID", fontsize=9)
    ax.set_xlabel("Run Index", fontsize=9)
    ax.tick_params(labelsize=8)
    
    # Legend condensed for small width
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), 
              frameon=False, fontsize=7, ncol=4)
    
    sns.despine(ax=ax)
    # Save with 300 DPI and embedded fonts
    plt.savefig("figure1_timeline.pdf", dpi=300)
    plt.show()

def plot_pca_polar(results, active_clusters, ch_names, color_map):
    """
    Figure 2: PCA Polar Plots (Full-page width: 170mm, 3 per row)
    Features: 
    - Centered bottom row with custom spacing.
    - Subtitle with color-coded line-style keys and variance %.
    """
    n_k = len(active_clusters)
    if n_k == 0: return

    cols = 3
    rows = (n_k + cols - 1) // cols
    
    width_in = 170 * MM_TO_INCH
    height_in = (rows * 70) * MM_TO_INCH # Increased slightly for manual text positioning
    
    fig = plt.figure(figsize=(width_in, height_in))
    
    gs = GridSpec(rows, cols, figure=fig, wspace=0.4)
    
    angles = np.linspace(0, 2 * np.pi, len(ch_names), endpoint=False).tolist()
    angles += angles[:1]
    
    line_symbols = ["—", "--", "···"]
    
    for i, k in enumerate(active_clusters):
        curr_row = i // cols
        curr_col = i % cols
        
        is_last_row = curr_row == rows - 1
        num_in_last_row = n_k % cols
        
        if is_last_row and num_in_last_row != 0:
            if num_in_last_row == 1:
                ax = fig.add_subplot(gs[curr_row, 1], polar=True)
            else: 
                sub_gs = gs[curr_row, :].subgridspec(1, 4, width_ratios=[0.2, 1, 1, 0.2], wspace=0.8)
                ax = fig.add_subplot(sub_gs[0, curr_col + 1], polar=True)
        else:
            ax = fig.add_subplot(gs[curr_row, curr_col], polar=True)

        # --- Plotting Logic ---
        data = results[k]
        c = color_map.get(str(k), 'black')
        line_color = c if k != -1 else 'gray' # Color for both plot and title text
        
        widths = [1.8, 1.2, 0.8]
        styles = ['-', '--', ':']
        
        # We will build the subtitle manually using ax.text
        pc_texts = []
        
        for pc_idx, component in enumerate(data['pca']):
            vec = component['vector'].tolist()
            vec += vec[:1]
            var_pct = component['variance'] * 100
            
            # Store strings for manual rendering
            pc_texts.append(f"{line_symbols[pc_idx]} {var_pct:.1f}%")
            
            ax.plot(angles, vec, color=line_color, 
                    linestyle=styles[pc_idx], linewidth=widths[pc_idx])
            
            if pc_idx == 0:
                ax.fill(angles, vec, color=line_color, alpha=0.1)

        # --- Formatting for Publication ---
        ax.set_xticks(angles[:-1])
        ax.tick_params(pad=-3)
        ax.set_xticklabels(ch_names, fontsize=6.5, weight='bold') 
        ax.set_yticks([]) 
        
        # 1. Main Title (Black)
        ax.set_title(f"Cluster {k}", fontsize=10, weight='bold', pad=25)
        
        # 2. Manual Subtitle (Colored)
        # We use a single string but color the whole line to match the cluster color.
        # To have different colors for different PCs, you'd need multiple text calls, 
        # but matching the cluster color is the standard publication look.
        var_subtitle = "    ".join(pc_texts)
        ax.text(0.5, 1.17, f"({var_subtitle})", 
                transform=ax.transAxes, 
                fontsize=8, 
                color=line_color, 
                weight='bold', 
                ha='center', 
                va='center')

    plt.savefig("figure2_polar_colored_titles.pdf", dpi=300, bbox_inches='tight')
    plt.show()

def plot_compact_stability(results, active_clusters, color_map):
    """
    Figure 3: Compact Stability with Top-Aligned Stats
    """
    plot_data = []
    for k in active_clusters:
        scores = results[k]['stability_scores']
        for s in scores:
            plot_data.append({'Cluster': str(k), 'Similarity': s})
            
    df_stab = pd.DataFrame(plot_data)
    
    width_in = 85 * MM_TO_INCH
    height_in = 70 * MM_TO_INCH
    
    # Increase the top margin slightly to make room for labels
    fig, ax = plt.subplots(figsize=(width_in, height_in), constrained_layout=True)
    
    sns.boxplot(data=df_stab, x='Cluster', y='Similarity', ax=ax, 
                palette=color_map, width=0.5, linewidth=1.0, 
                showfliers=False) 

    # Calculate statistics
    stats = df_stab.groupby('Cluster')['Similarity'].agg(['count', 'median'])
    x_labels = [t.get_text() for t in ax.get_xticklabels()]
    
    # Iterate through each cluster to place labels at the top
    for i, label in enumerate(x_labels):
        if label in stats.index:
            n_obs = stats.loc[label, 'count']
            med_val = stats.loc[label, 'median']
            
            # Place Median (Top Line)
            # transform=ax.get_xaxis_transform() allows X to be data coords and Y to be 0-1 (axes)
            ax.text(i, 1.08, f'M: {med_val:.2f}', 
                    transform=ax.get_xaxis_transform(),
                    ha='center', va='center', fontsize=7, fontweight='bold')
            
            # Place Sample Size (Second Line)
            ax.text(i, 1.02, f'n = {n_obs}', 
                    transform=ax.get_xaxis_transform(),
                    ha='center', va='center', fontsize=7, color='dimgray')

    ax.set_ylabel("Similarity", fontsize=9)
    ax.set_xlabel("Cluster ID", fontsize=9)
    ax.tick_params(labelsize=8)
    
    # Set y-limit to 1.0 for the data, but the labels sit above this in the margin
    ax.set_ylim(0, 1.0)
    
    # Clip-off is disabled so text outside the 0-1 range is still visible
    sns.despine(ax=ax)
    plt.savefig("figure3_stability.pdf", dpi=300, bbox_inches='tight')
    plt.show()


def analyze_stability_wilcoxon(results, active_clusters, color_map):
    """
    Figure 3: Stability Analysis with Wilcoxon Post-hoc
     - Global Test: Kruskal-Wallis to check for overall differences.
     - Post-hoc: Mann-Whitney U with Bonferroni correction for pairwise comparisons.
     - Visualization: Boxplot with significance brackets above the plot.
    """
    # 1. Data Preparation
    valid_clusters = [k for k in active_clusters if k in results]
    data_list = [{'Cluster': str(k), 'Similarity': score} 
                 for k in valid_clusters 
                 for score in results[k]['stability_scores']]
    
    df = pd.DataFrame(data_list)
    if df.empty or len(valid_clusters) < 2: 
        return

    # 2. Global Test
    groups = [results[k]['stability_scores'] for k in valid_clusters]
    h_stat, p_global = kruskal(*groups)
    
    if p_global >= 0.05:
        print(f"Global test non-sig: p={p_global:.4f}")
        return

    # 3. Post-hoc: Wilcoxon with Bonferroni
    pairs = list(itertools.combinations(valid_clusters, 2))
    n_comparisons = len(pairs)
    wilcoxon_results = {}
    
    for c1, c2 in pairs:
        _, p_raw = mannwhitneyu(results[c1]['stability_scores'], 
                                results[c2]['stability_scores'], 
                                alternative='two-sided')
        p_adj = min(p_raw * n_comparisons, 1.0) 
        wilcoxon_results[(str(c1), str(c2))] = p_adj

    # 4. Visualization Setup
    # Using 170mm for full-width or 85mm for half-width
    width_in = 170 * MM_TO_INCH 
    height_in = 100 * MM_TO_INCH
    
    fig, ax = plt.subplots(figsize=(width_in, height_in))
    
    # Boxplot with thinner lines for publication style
    # Whiskers should denote the 75th and 25th percentiles
    sns.boxplot(data=df, x='Cluster', y='Similarity', ax=ax, 
                palette=color_map, width=0.2, linewidth=0.75, showfliers=False, vert=True)
    sns.stripplot(data=df, x='Cluster', y='Similarity', ax=ax, 
                  color='black', alpha=0.3, size=2, jitter=0.2)
    
    # 5. Significance Brackets
    # Dynamic scaling based on the data range
    y_range = df['Similarity'].max() - df['Similarity'].min()
    bracket_base = df['Similarity'].max() + (y_range * 0.05)
    step = y_range * 0.12
    
    labels = [str(k) for k in valid_clusters]
    significant_count = 0

    for i, c1 in enumerate(labels):
        for j, c2 in enumerate(labels):
            if i >= j: continue
            p_val = wilcoxon_results.get((c1, c2))
            
            if p_val and p_val < 0.05:
                stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*"
                
                # Calculate vertical position for this specific bracket
                y = bracket_base + (significant_count * step)
                h = step * 0.2 # height of the "tick" on the bracket
                
                ax.plot([i, i, j, j], [y, y+h, y+h, y], lw=0.75, color='black')
                ax.text((i+j)*0.5, y+h, stars, ha='center', va='bottom', 
                        fontsize=8, fontweight='bold')
                significant_count += 1

    # Labels and Aesthetics
    ax.set_ylabel("Cosine Similarity\n(Most Dominant Eigenvector)", fontsize=9)
    ax.set_xlabel("Cluster ID", fontsize=9)
    
    # Adjust ylim to fit all brackets
    ax.set_ylim(df['Similarity'].min() - 0.05, bracket_base + (significant_count * step) + 0.1)
    
    sns.despine()
    plt.tight_layout()
    plt.savefig("figure3_stability.pdf", dpi=300)
    plt.show()

In [ ]:
def fixed_cluster_colors(labels):
    """
    Assigns consistent colors to clusters using STRING keys for safety.
    """
    base_colors = [
        (0.84, 0.15, 0.16, 1.0),  # red
        (0.12, 0.47, 0.71, 1.0),  # blue
        (0.17, 0.63, 0.17, 1.0),  # green
        (0.58, 0.40, 0.74, 1.0),  # purple
        (1.00, 0.50, 0.05, 1.0),  # orange
    ]

    cluster_color = {}
    unique_labels = np.unique(labels.astype(int))
    color_idx = 0
    
    for lb in unique_labels:
        if lb == -1:
            cluster_color[str(lb)] = (0.5, 0.5, 0.5, 1.0) # Grey for Noise
            continue
            
        cluster_color[str(lb)] = base_colors[color_idx % len(base_colors)]
        color_idx += 1

    return cluster_color

# ==========================================
# 1. USER INPUT SECTION
# ==========================================

def load_user_data():
    
    try:
        centroid_mat = scipy.io.loadmat(data_path / "<filename>.mat")
        raw_centroids = centroid_mat["run_centroids"][0, :, :, :]
        run_centroids = np.concatenate([raw_centroids[:, 0, :, :], raw_centroids[:, 1, :, :]], axis=0)
        print("Loaded centroid data shape:", run_centroids.shape)
    except FileNotFoundError:
        print("Error: Centroid file not found. Generating Mock Data.")
        run_centroids = np.random.randn(100, 13, 13)
        run_centroids = np.matmul(run_centroids, run_centroids.transpose(0, 2, 1))

    try:
        cluster_labels = np.load(data_path / 'cluster_labels.npy')[0, :] 
        cluster_labels = np.concatenate([cluster_labels, cluster_labels], axis=0)
        cluster_labels = cluster_labels.astype(int) 
        print("Loaded cluster labels shape:", cluster_labels.shape)
    except FileNotFoundError:
        print("Error: Label file not found. Generating Mock Labels.")
        cluster_labels = np.random.choice([-1, 0, 1, 2], size=len(run_centroids))

    time_indices = np.arange(len(cluster_labels))
    color = fixed_cluster_colors(cluster_labels)
    
    channel_names =  [
        "FC3", "FC1", "FCZ", "FC2", "FC4",
        "C3", "C1", "CZ", "C2", "C4",
        "CP1", "CPZ", "CP2",
    ]

    return run_centroids, cluster_labels, time_indices, channel_names, color

# ==========================================
# 2. ANALYSIS LOGIC
# ==========================================

def get_eigen_details(cov):
    vals, vecs = np.linalg.eigh(cov)
    idx = np.argsort(vals)[::-1]
    return vals[idx], vecs[:, idx]

def get_pca_components(cov, n_components=3):
    vals, vecs = get_eigen_details(cov)
    total_var = np.sum(vals)
    explained_var = vals / total_var

    printable_var = [round(v, 2) for v in explained_var]
    print(printable_var)
    
    components = []
    for i in range(n_components):
        components.append({
            'vector': np.abs(vecs[:, i]),
            'variance': explained_var[i]
        })
    return components

def perform_scree_test(centroid, cluster_id):
    """
    Analyzes the eigenvalue decay to justify dimensionality reduction.
    """
    vals, _ = get_eigen_details(centroid)
    total_var = np.sum(vals)
    cum_var = np.cumsum(vals) / total_var
    
    # Simple elbow detection: where the drop in explained variance becomes < 5%
    diffs = np.diff(vals / total_var)
    elbow_index = np.where(np.abs(diffs) < 0.05)[0][0] + 1 if any(np.abs(diffs) < 0.05) else 3
    
    print(f"\n--- Scree Analysis for Cluster {cluster_id} ---")
    print(f"Top 3 components explain: {cum_var[2]:.1%} of total variance.")
    print(f"Mathematical elbow detected at component: {elbow_index}")
    
    if cum_var[2] > 0.8:
        print("Justification: The 3-component model captures >80% of signal energy.")
    
    return vals, cum_var, elbow_index

def perform_cluster_stats(results, clusters):
    """
    Checks if the Stability (Cosine Similarity) distributions are 
    statistically different across clusters.
    """
    # 1. Prepare data groups
    valid_clusters = [k for k in clusters if k in results]
    groups = [results[k]['stability_scores'] for k in valid_clusters]
    
    if len(groups) < 2:
        print("Not enough clusters for statistical comparison.")
        return

    # 2. Global Test: Kruskal-Wallis
    h_stat, p_global = kruskal(*groups)
    
    print("-" * 50)
    print(f"Global Test (Kruskal-Wallis): H={h_stat:.3f}, p={p_global:.4e}")
    
    if p_global >= 0.05:
        print("Result: No significant differences found.")
        return

    print("Result: Significant differences detected! Performing pairwise tests...")
    print("-" * 50)

    # 3. Post-hoc: Pairwise Mann-Whitney U with Bonferroni
    pairs = list(itertools.combinations(valid_clusters, 2))
    n_comparisons = len(pairs)
    
    for c1, c2 in pairs:
        s1 = results[c1]['stability_scores']
        s2 = results[c2]['stability_scores']
        
        _, p_raw = mannwhitneyu(s1, s2, alternative='two-sided')
        
        # Apply correction: p_adj = p_raw * total_tests
        p_adj = min(p_raw * n_comparisons, 1.0)
        
        verdict = "Significant" if p_adj < 0.05 else "Not Significant"
        print(f"Cluster {c1} vs {c2}: p_adj = {p_adj:.4f} ({verdict})")

def analyze_clusters_full(X, y):
    unique_clusters = np.sort(np.unique(y))
    indices = [0, 1, 2, 3, 4]
    cluster_centroids = []
    n_covs_cluster = []
    colors = ["NOISE", "RED", "BLUE", "GREEN", "PURPLE"]
    print(unique_clusters)
    results = {}

    
    for k in unique_clusters:
        cluster_covs = X[y == k]
        if len(cluster_covs) == 0: continue

        centroid = mean_riemann(cluster_covs)
        cluster_centroids.append(centroid)
        n_covs_cluster.append(len(cluster_covs))
        print(f"Cluster {k} Centroid Eigenvalues and Eigenvectors:")
        pca_data = get_pca_components(centroid, n_components=3)

        vals, cum_var, elbow = perform_scree_test(centroid, k)
        
        _, vecs_c = get_eigen_details(centroid)
        pc1_centroid = vecs_c[:, 0]
        
        similarities = []
        for cov in cluster_covs:
            _, vecs_i = get_eigen_details(cov)
            pc1_sample = vecs_i[:, 0]
            # sim = np.abs(np.dot(pc1_sample, pc1_centroid))
            sim = np.abs(np.dot(pc1_sample, pc1_centroid) / (np.linalg.norm(pc1_sample) * np.linalg.norm(pc1_centroid)))
            similarities.append(sim)

        print(f"Cluster {k} Stability Scores (Cosine Similarity to Centroid PC1):")
        print(f"  Median: {np.median(similarities):.4f}, Mean: {np.mean(similarities):.4f}, Std: {np.std(similarities):.4f}, Min: {np.min(similarities):.4f}, Max: {np.max(similarities):.4f}")
        print(f" Number of samples in Cluster {k}: {len(cluster_covs)}")
        
        results[k] = {
            'pca': pca_data,
            'stability_scores': similarities,
            'n_samples': len(cluster_covs)
        }
    
    # Compute the normalized distance between the cluster centroids
    for idx_1, idx_2 in itertools.combinations(indices, 2):
        c1 = cluster_centroids[idx_1]
        c2 = cluster_centroids[idx_2]

        dist = distance_riemann(c1, c2)
        
        n_covs_c1 = n_covs_cluster[idx_1]
        n_covs_c2 = n_covs_cluster[idx_2]

        dist = dist / (1/2 * (n_covs_c1 + n_covs_c2)) # Normalized distance

        print(f"Normalized Riemannian Distance between Cluster Centroids: {colors[idx_1]} vs {colors[idx_2]} = {dist:.4f}")

        
    return results, unique_clusters

def print_cluster_report(results):
    print("\n" + "="*85)
    print(f"{'CLUSTER METRICS REPORT':^85}")
    print("="*85)
    print(f"{'ID':<6} | {'N':<5} | {'Purity (PC1)':<15} | {'Stability (Med)':<18} | {'Verdict':<20}")
    print("-" * 85)
    
    clusters = sorted(results.keys())
    
    for k in clusters:
        data = results[k]
        n = data['n_samples']
        purity = data['pca'][0]['variance']
        stability = np.median(data['stability_scores'])
        
        verdict = ""
        if k == -1: verdict = "⚫ NOISE / ARTIFACT"
        elif purity > 0.70 and stability > 0.90: verdict = "✅ ROBUST STATE"
        elif stability > 0.90: verdict = "🔹 STABLE (COMPLEX)"
        elif purity > 0.70: verdict = "⚠️ DRIFTING SOURCE"
        else: verdict = "❌ UNSTABLE"
            
        print(f"{k:<6} | {n:<5} | {purity:.1%} {'(High)' if purity>0.7 else '(Low )':<7} | {stability:.3f} {'(Tight)' if stability>0.9 else '(Loose)':<8} | {verdict:<20}")
    print("="*85 + "\n")

In [ ]:
data, labels, time_idx, ch_names, color_map = load_user_data()

# 1. Analyze
results, clusters = analyze_clusters_full(data, labels)

# 2. Print Report
print_cluster_report(results)

# 3. Statistical Significance Testing (Cluster Stability Difference)
# perform_cluster_stats(results, clusters)
print("Generating Stability Plot...")
# analyze_and_plot_stability(results, clusters, color_map)
analyze_stability_wilcoxon(results, clusters, color_map)

# 4. Plot 2: PCA Polar (Extracted & Beautified)
print("Generating PCA Polar Plots...")
plot_pca_polar(results, clusters, ch_names, color_map)

# 5. Plot 3: Stability
# print("Generating Stability Plot...")
# plot_compact_stability(results, clusters, color_map)
